# PTI Algorithm Notebook: Piano Transcription to MIDI

This notebook explains and visualizes the **algorithmic core** of the PTI-based transcription workflow and how events are handled in this application.

Focus:
- PTI model theory (heads + decoding)
- Synthetic decoder demonstration
- App-level postprocess effects on note/pedal events

Out of scope:
- YouTube ingestion and deployment workflows


## 1) Learning Goals

By the end, you should be able to:
1. Describe PTI's multi-head formulation for piano transcription.
2. Explain how regression outputs are decoded into note/pedal events.
3. Understand how this app maps PTI output to MIDI and postprocesses events.
4. Interpret the impact of postprocess parameters on final note events.


In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
from statistics import median

import numpy as np
import matplotlib.pyplot as plt

from audio2midi.models import NoteEvent, PedalEvent, TranscriptionResult
from audio2midi.postprocess import postprocess_transcription

plt.style.use("seaborn-v0_8")


In [ ]:
# Notebook config contract
RUN_SYNTHETIC_DECODER_DEMO = True
RUN_OPTIONAL_HEAVY_PTI = False
USE_CACHED_ARTIFACTS = True

# Optional local WAV path for heavy PTI demo
AUDIO_WAV_PATH = None  # e.g. ".cache/audio2midi/preprocessed/UAOtlGu2aKA.wav"
PTI_CHECKPOINT_PATH = None  # e.g. "/path/to/note_F1=0.9677_pedal_F1=0.9186.pth"

# App-level postprocess parameters
MIN_DURATION_SECONDS = 0.05
NOTE_THRESHOLD = 0.50
QUANTIZE_GRID_SECONDS = None

FPS = 100  # PTI frames-per-second


## 2) Task Formulation

For piano AMT, we estimate event sets from waveform \(x(t)\):

- Note events: \((pitch, onset, offset, velocity)\)
- Sustain pedal events: \((onset, offset)\)

Challenges:
- Polyphony: multiple keys active simultaneously.
- Timing precision: onset/offset boundaries matter.
- Sustain: pedal alters effective note duration.


## 3) PTI Architecture (Theory)

PTI (High-resolution Piano Transcription with Pedals) uses:

1. **Shared acoustic front-end**
- Input sample rate: **16 kHz**
- STFT: **n_fft=2048**
- Frames-per-second: **100**
- Log-mel bins: **229**

2. **Note heads (88 keys)**
- Regression onset
- Regression offset
- Frame activity
- Velocity

3. **Pedal heads**
- Pedal onset regression
- Pedal offset regression
- Pedal frame output

4. **Decoder**
- Converts regression/frame outputs to event tuples with sub-frame shift corrections.


## 4) Decoder Theory and Pseudo-code

PTI decoder behavior in words:

1. Binarize regression outputs using threshold + local monotonic neighborhood checks.
2. Detect note onsets first.
3. For each onset, find note ending via frame-disappear and/or offset signal.
4. Apply onset/offset shift terms for sub-frame timing.
5. Decode pedal intervals similarly with pedal frame/offset signals.

Pseudo-code (single-pitch view):

```text
for t in frames:
    if onset[t] detected:
        start = t
    if active note:
        monitor frame and offset
        choose end by decoder rule
        emit (start, end, onset_shift, offset_shift, velocity)
```


In [ ]:
# 5) Synthetic decoder demo using PTI VAD functions

pti_vad_available = True
try:
    from piano_transcription_inference.piano_vad import (
        note_detection_with_onset_offset_regress,
        pedal_detection_with_onset_offset_regress,
    )
except Exception as exc:
    pti_vad_available = False
    pti_vad_import_error = exc

frames = 320
frame_threshold = 0.1

# Single-pitch synthetic outputs (for theory demonstration)
onset_output = np.zeros(frames)
onset_shift_output = np.zeros(frames)
offset_output = np.zeros(frames)
offset_shift_output = np.zeros(frames)
frame_output = np.zeros(frames)
velocity_output = np.zeros(frames)

# Define three synthetic notes
synthetic_note_spans = [(40, 85, 0.80), (120, 168, 0.65), (210, 260, 0.90)]
for idx, (bgn, fin, vel) in enumerate(synthetic_note_spans):
    onset_output[bgn] = 1
    onset_shift_output[bgn] = [0.15, -0.10, 0.05][idx]
    offset_output[fin] = 1
    offset_shift_output[fin] = [0.20, -0.05, 0.10][idx]
    frame_output[bgn:fin + 1] = np.maximum(frame_output[bgn:fin + 1], 0.75)
    velocity_output[bgn] = vel

# Synthetic pedal activity
pedal_frame_output = np.zeros(frames)
pedal_offset_output = np.zeros(frames)
pedal_offset_shift_output = np.zeros(frames)
pedal_frame_output[35:175] = 0.9
pedal_offset_output[175] = 1
pedal_offset_shift_output[175] = 0.1

pedal_frame_output[200:285] = 0.85
pedal_offset_output[285] = 1
pedal_offset_shift_output[285] = -0.1

if pti_vad_available and RUN_SYNTHETIC_DECODER_DEMO:
    synthetic_note_tuples = note_detection_with_onset_offset_regress(
        frame_output=frame_output,
        onset_output=onset_output,
        onset_shift_output=onset_shift_output,
        offset_output=offset_output,
        offset_shift_output=offset_shift_output,
        velocity_output=velocity_output,
        frame_threshold=frame_threshold,
    )
    synthetic_pedal_tuples = pedal_detection_with_onset_offset_regress(
        frame_output=pedal_frame_output,
        offset_output=pedal_offset_output,
        offset_shift_output=pedal_offset_shift_output,
        frame_threshold=frame_threshold,
    )
else:
    synthetic_note_tuples = []
    synthetic_pedal_tuples = []

print("PTI VAD import:", "ok" if pti_vad_available else f"missing ({pti_vad_import_error})")
print("Synthetic note tuples:", synthetic_note_tuples)
print("Synthetic pedal tuples:", synthetic_pedal_tuples)


In [ ]:
# 6) Visualize decoder inputs/outputs for the synthetic single-pitch case

def to_seconds(frame_index: float, fps: int = FPS) -> float:
    return frame_index / fps

time_axis = np.arange(frames) / FPS
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(time_axis, frame_output, label="frame_output")
axes[0].set_ylabel("Frame")
axes[0].axhline(frame_threshold, linestyle="--", linewidth=1, label="threshold")
axes[0].legend(loc="upper right")

axes[1].plot(time_axis, onset_output, label="onset_output")
axes[1].plot(time_axis, offset_output, label="offset_output")
axes[1].set_ylabel("On/Off")
axes[1].legend(loc="upper right")

axes[2].plot(time_axis, velocity_output, label="velocity_output", color="tab:orange")
axes[2].set_ylabel("Velocity")
axes[2].legend(loc="upper right")

axes[3].plot(time_axis, pedal_frame_output, label="pedal_frame_output", color="tab:green")
axes[3].plot(time_axis, pedal_offset_output, label="pedal_offset_output", color="tab:red")
axes[3].set_ylabel("Pedal")
axes[3].set_xlabel("Time (s)")
axes[3].legend(loc="upper right")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle("Synthetic PTI Decoder Signals (Single-Pitch Demonstration)")
plt.tight_layout()
plt.show()


In [ ]:
# 7) Convert synthetic tuples into app event types

synthetic_notes: list[NoteEvent] = []
synthetic_pedals: list[PedalEvent] = []

# In this single-pitch synthetic demo, we attach all events to demo pitch 60.
DEMO_PITCH = 60

for tup in synthetic_note_tuples:
    bgn, fin, onset_shift, offset_shift, norm_vel = tup
    start = (bgn + onset_shift) / FPS
    end = (fin + offset_shift) / FPS
    velocity = int(max(1, min(127, round(float(norm_vel) * 127))))
    synthetic_notes.append(
        NoteEvent(
            pitch=DEMO_PITCH,
            start=float(start),
            end=float(end),
            velocity=velocity,
            confidence=None,
        )
    )

for tup in synthetic_pedal_tuples:
    bgn, fin, onset_shift, offset_shift = tup
    start = (bgn + onset_shift) / FPS
    end = (fin + offset_shift) / FPS
    synthetic_pedals.append(
        PedalEvent(
            start=float(start),
            end=float(end),
            value=127,
        )
    )

raw_result = TranscriptionResult(notes=synthetic_notes, pedals=synthetic_pedals)
print("Raw synthetic events:")
print("- notes:", len(raw_result.notes))
print("- pedals:", len(raw_result.pedals))
for n in raw_result.notes:
    print(asdict(n))


## 5) Application Mapping: PTI -> MIDI -> App Postprocess

In this repository, PTI is used as a backend transcriber. The app-level flow is:

1. `create_transcriber("pti", ...)`
2. PTI backend transcribes to MIDI internally.
3. App reads that MIDI into `TranscriptionResult` (`read_midi_as_transcription`).
4. App applies `postprocess_transcription` for filtering, pedal extension, and optional quantization.

Important caveat:
- PTI raw head tensors are not exposed by this app's production pipeline by default.


In [ ]:
# 8) Postprocess sweep visualization

sweep_settings = [
    (0.03, 0.40, None),
    (0.05, 0.50, None),
    (0.08, 0.60, 0.02),
]

rows = []
for min_dur, threshold, grid in sweep_settings:
    cleaned = postprocess_transcription(
        raw_result,
        min_duration_seconds=min_dur,
        note_on_threshold=threshold,
        quantize_grid_seconds=grid,
    )
    durations = [n.end - n.start for n in cleaned.notes]
    rows.append(
        {
            "min_duration": min_dur,
            "note_threshold": threshold,
            "quantize_grid": grid,
            "note_count": len(cleaned.notes),
            "pedal_count": len(cleaned.pedals),
            "median_note_duration": median(durations) if durations else 0.0,
        }
    )

print("Postprocess sweep summary")
for row in rows:
    print(row)

cleaned_result = postprocess_transcription(
    raw_result,
    min_duration_seconds=MIN_DURATION_SECONDS,
    note_on_threshold=NOTE_THRESHOLD,
    quantize_grid_seconds=QUANTIZE_GRID_SECONDS,
)
print("
Default cleaned events:", len(cleaned_result.notes), "notes")


In [ ]:
# 9) Event visualizations: piano roll, velocity, pedal timeline

def plot_piano_roll(result: TranscriptionResult, title: str):
    fig, ax = plt.subplots(figsize=(14, 4))
    for note in result.notes:
        ax.barh(
            y=note.pitch,
            width=note.end - note.start,
            left=note.start,
            height=0.7,
            alpha=0.7,
        )
    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("MIDI Pitch")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_velocity_hist(result: TranscriptionResult, title: str):
    velocities = [n.velocity for n in result.notes]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    if velocities:
        ax.hist(velocities, bins=16, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Velocity")
    ax.set_ylabel("Count")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_pedal_timeline(result: TranscriptionResult, title: str):
    fig, ax = plt.subplots(figsize=(14, 2.2))
    for i, pedal in enumerate(result.pedals):
        ax.barh(
            y=0,
            width=pedal.end - pedal.start,
            left=pedal.start,
            height=0.6,
            alpha=0.6,
            label="pedal" if i == 0 else None,
        )
    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_yticks([])
    ax.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_piano_roll(raw_result, "Raw Synthetic Events (Single-Pitch Demo)")
plot_piano_roll(cleaned_result, "Cleaned Synthetic Events")
plot_velocity_hist(cleaned_result, "Velocity Distribution")
plot_pedal_timeline(cleaned_result, "Pedal Timeline")


In [ ]:
# 10) Optional heavy PTI run (guarded)

if RUN_OPTIONAL_HEAVY_PTI:
    if AUDIO_WAV_PATH is None:
        print("Set AUDIO_WAV_PATH before running heavy PTI mode.")
    else:
        from audio2midi.transcribers.base import create_transcriber

        checkpoint_path = Path(PTI_CHECKPOINT_PATH).expanduser().resolve() if PTI_CHECKPOINT_PATH else None
        transcriber = create_transcriber(
            backend="pti",
            device="cpu",
            pti_checkpoint_path=checkpoint_path,
        )
        heavy_raw = transcriber.transcribe(Path(AUDIO_WAV_PATH))
        heavy_clean = postprocess_transcription(
            heavy_raw,
            min_duration_seconds=MIN_DURATION_SECONDS,
            note_on_threshold=NOTE_THRESHOLD,
            quantize_grid_seconds=QUANTIZE_GRID_SECONDS,
        )
        print("Heavy PTI result:", len(heavy_clean.notes), "notes", len(heavy_clean.pedals), "pedals")
        plot_piano_roll(heavy_clean, "Heavy PTI: Cleaned Piano Roll")
        plot_velocity_hist(heavy_clean, "Heavy PTI: Velocity Distribution")
        plot_pedal_timeline(heavy_clean, "Heavy PTI: Pedal Timeline")
else:
    print("Heavy PTI mode skipped. Set RUN_OPTIONAL_HEAVY_PTI=True to run.")


## 6) Accuracy Caveats

1. PTI internals produce rich head outputs, but the app's production flow primarily consumes PTI-generated MIDI and then applies app-level postprocessing.
2. This synthetic demo uses PTI VAD functions on a single-pitch signal to illustrate decoding mechanics clearly.
3. In real audio, decoding occurs per pitch and event sets are merged across all 88 notes plus pedal events.


## 7) Conclusion

Key algorithm levers in this application:
- `MIN_DURATION_SECONDS`: suppresses short spurious notes.
- `NOTE_THRESHOLD`: confidence filter when confidence is available.
- `QUANTIZE_GRID_SECONDS`: optional timing regularization.
- Pedal-aware extension in postprocess preserves sustain behavior.

This notebook showed both:
- PTI theory and decoding logic,
- and the concrete app-level event transformations applied before final MIDI writing.


## References

- PTI paper: https://arxiv.org/abs/2010.01815
- PTI code: https://github.com/bytedance/piano_transcription
- Repository modules:
- `audio2midi/transcribers/piano_transcription_inference.py`
- `audio2midi/postprocess.py`
- `audio2midi/midi_writer.py`
- `audio2midi/models.py`
